[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/32_topk_sampling.ipynb)

# 🟠 Medium: Top-k / Top-p (Nucleus) Sampling

Implement **sampling with top-k and top-p filtering** — the standard LLM decoding strategy.

### Signature
```python
def sample_top_k_top_p(logits, top_k=0, top_p=1.0, temperature=1.0) -> int:
    # logits: (V,) unnormalized log-probabilities
    # Returns: sampled token index
```

### Algorithm
1. Scale by temperature: `logits /= temperature`
2. Top-k: keep only top-k logits, set rest to `-inf`
3. Top-p: sort by prob, mask tokens where cumulative prob exceeds p
4. Sample from filtered distribution

In [2]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [3]:
import torch

In [63]:
# ✏️ YOUR IMPLEMENTATION HERE

def sample_top_k_top_p(logits, top_k=0, top_p=1.0, temperature=1.0):
    # pass  # temperature, top-k filter, top-p filter, sample
    logits /= temperature
    if top_k > 0:
      topk_vals, topk_idxs = torch.topk(logits, top_k)
      mask = torch.zeros_like(logits, dtype = torch.bool)
      # mask.scatter_(dim, indices, True) sets mask[i, indices[i, j]] = True for each j along dim.
      mask = mask.scatter(0, topk_idxs, True)
      logits = logits.masked_fill(~mask, -float('inf'))
    elif top_p < 1.0:
      prob = torch.softmax(logits, 0)
      prob = torch.sort(prob).values
      cumu_prob = torch.cumsum(prob, -1)
      mask = (cumu_prob <= top_p).bool()
      logits = logits.masked_fill(~mask, -float('inf'))

    sample_prob = torch.softmax(logits, -1)
    return torch.multinomial(sample_prob, 1).item()


In [64]:
# 🧪 Debug
logits = torch.tensor([1.0, 5.0, 2.0, 0.5])
print('top_k=1:', sample_top_k_top_p(logits.clone(), top_k=1))
print('top_p=0.5:', sample_top_k_top_p(logits.clone(), top_p=0.5))
print('temp=0.01:', sample_top_k_top_p(logits.clone(), temperature=0.01))

top_k=1: 1
top_p=0.5: 1
temp=0.01: 1


In [65]:
# ✅ SUBMIT
from torch_judge import check
check('topk_sampling')


🧪 Testing: Top-k / Top-p Sampling (Medium)
──────────────────────────────────────────────────
  ✅ [1/4] top_k=1 always returns argmax (10.4ms)
  ✅ [2/4] Low temperature concentrates (16.6ms)
  ✅ [3/4] All tokens reachable (no filtering) (331.2ms)
  ✅ [4/4] Returns valid index (6.7ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (364.8ms total)
  Progress saved. Run status() to see your dashboard.

